# 🧬 Quantum Protein Folding
### Quantum for Healthcare — Quantum for Humanity

This notebook demonstrates **quantum optimisation for protein folding** using the lattice protein model — a step toward understanding disease-causing misfolded proteins (Alzheimer's, Parkinson's, Prion disease).

**Learning source:** [IBM Quantum — Protein Folding](https://github.com/qiskit-community/qiskit-research/tree/main/docs/protein_folding)

---

## Background

**Protein folding** determines the 3D structure of a protein from its amino acid sequence — and thus its biological function. Misfolded proteins cause:
- Alzheimer's disease (amyloid-β misfolding)
- Parkinson's disease (α-synuclein aggregation)
- Type 2 Diabetes (IAPP misfolding)
- Mad Cow disease / Prion diseases

Protein folding is **NP-hard** — finding the minimum energy conformation in the space of all possible 3D structures. Quantum optimisation (QAOA) offers a new approach.

We use the **HP (Hydrophobic-Polar) lattice model** — a simplified but widely used model.

In [ ]:
# Install required packages (run once)
# !pip install qiskit qiskit-algorithms qiskit-optimization

import numpy as np
import matplotlib.pyplot as plt
from itertools import product

# Quantum imports
from qiskit_algorithms import QAOA, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit.primitives import Sampler

print('✅ Imports successful')

## Step 1: Define the HP Lattice Protein Model

In [ ]:
# HP Model: each amino acid is either Hydrophobic (H) or Polar (P)
# Energy: -1 for each H-H contact in 3D that are not adjacent in sequence
# Goal: minimise total energy (maximise H-H contacts)

# Short peptide sequence (H=Hydrophobic, P=Polar)
# Example: fragment of amyloid-β peptide (simplified)
sequence = 'HPPHPPHHP'   # 9 amino acids
n = len(sequence)

print(f'Protein sequence: {sequence}')
print(f'Length: {n} amino acids')
print(f'H residues at positions: {[i for i,s in enumerate(sequence) if s=="H"]}')
print(f'P residues at positions: {[i for i,s in enumerate(sequence) if s=="P"]}')

# Visual representation
print('\nSequence:')
print('  ' + '─'.join(f'[{s}]' for s in sequence))
print('  ' + '   '.join(f' {i} ' for i in range(n)))

## Step 2: Enumerate Conformations on a 2D Lattice

In [ ]:
# For the 2D HP lattice model, encode turns between residues
# Direction encoding: 0=right, 1=up, 2=left, 3=down (modular)
# Each turn is encoded as a 2-bit binary variable

def place_protein(turns):
    """Place amino acids on 2D lattice given sequence of turns."""
    positions = [(0, 0)]
    dx = [1, 0, -1, 0]
    dy = [0, 1, 0, -1]
    direction = 0  # start going right
    for turn in turns:
        direction = (direction + turn) % 4
        x, y = positions[-1]
        positions.append((x + dx[direction], y + dy[direction]))
    return positions

def compute_energy(positions, sequence):
    """Compute HP energy: -1 per non-bonded H-H contact."""
    energy = 0
    pos_set = {pos: i for i, pos in enumerate(positions)}
    for i, (x, y) in enumerate(positions):
        if sequence[i] == 'H':
            for nx, ny in [(x+1,y),(x-1,y),(x,y+1),(x,y-1)]:
                if (nx, ny) in pos_set:
                    j = pos_set[(nx, ny)]
                    if j != i-1 and j != i+1 and sequence[j] == 'H':
                        energy -= 0.5  # count each pair once
    return energy

def has_collision(positions):
    return len(set(positions)) < len(positions)

# Find best classical conformation by brute force (small sequences only)
best_energy = 0
best_turns  = None
best_positions = None

for turns in product([0, 1, 2, 3], repeat=n-1):
    pos = place_protein(turns)
    if not has_collision(pos):
        e = compute_energy(pos, sequence)
        if e < best_energy:
            best_energy = e
            best_turns = turns
            best_positions = pos

print(f'Classical best energy: {best_energy:.1f}')
print(f'Best turn sequence:    {best_turns}')

## Step 3: Visualise the Optimal Folding

In [ ]:
if best_positions:
    fig, ax = plt.subplots(figsize=(7, 7))
    xs = [p[0] for p in best_positions]
    ys = [p[1] for p in best_positions]
    
    # Draw backbone
    ax.plot(xs, ys, 'k-', linewidth=2, zorder=1, alpha=0.5)
    
    # Draw residues
    for i, (x, y) in enumerate(best_positions):
        color = '#8B5CF6' if sequence[i] == 'H' else '#D1D5DB'
        circle = plt.Circle((x, y), 0.35, color=color, zorder=2)
        ax.add_patch(circle)
        ax.text(x, y, sequence[i], ha='center', va='center', 
                fontsize=12, fontweight='bold', color='white' if sequence[i]=='H' else 'black', zorder=3)
    
    # Mark H-H contacts
    pos_map = {pos: i for i, pos in enumerate(best_positions)}
    drawn = set()
    for i, (x, y) in enumerate(best_positions):
        if sequence[i] == 'H':
            for nx, ny in [(x+1,y),(x-1,y),(x,y+1),(x,y-1)]:
                if (nx,ny) in pos_map:
                    j = pos_map[(nx,ny)]
                    if j != i-1 and j != i+1 and sequence[j]=='H' and (min(i,j),max(i,j)) not in drawn:
                        ax.plot([x,nx],[y,ny], 'r--', linewidth=2, alpha=0.6, zorder=1)
                        drawn.add((min(i,j),max(i,j)))
    
    ax.set_xlim(min(xs)-1, max(xs)+1)
    ax.set_ylim(min(ys)-1, max(ys)+1)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    ax.set_title(f'HP Protein Folding — Sequence: {sequence}\n'
                 f'Energy: {best_energy:.1f} (red dashes = H-H contacts)',
                 fontsize=12, fontweight='bold')
    
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#8B5CF6', label='Hydrophobic (H)'),
                       Patch(facecolor='#D1D5DB', label='Polar (P)')]
    ax.legend(handles=legend_elements, loc='upper right')
    plt.tight_layout()
    plt.show()

## Step 4: Encode as QUBO for QAOA

For larger proteins, encode the conformation search as a QUBO:
- Each turn encoded with 2 binary variables: $(t_{i,0}, t_{i,1})$
- Penalty terms enforce self-avoidance (no lattice collisions)
- Objective: maximise H-H contacts

```python
# Build QUBO for protein folding
from qiskit_optimization import QuadraticProgram

qp = QuadraticProgram('ProteinFolding')
# Add 2*(n-1) binary variables for n-1 turns
for i in range(n-1):
    qp.binary_var(f't{i}_0')
    qp.binary_var(f't{i}_1')

# Add objective (H-H contact energy) and collision penalty terms
# → Then run QAOA as in the finance notebooks
```

## 🌍 Humanitarian Impact

| Disease | Protein | Deaths/Year | Quantum Impact |
|---|---|---|---|
| Alzheimer's | Amyloid-β | 2M+ (dementia) | Design inhibitors of misfolding |
| Parkinson's | α-Synuclein | 100K+ | Aggregation inhibitor design |
| Malaria | PfCRT, PfDHFR | 600K | Drug resistance mechanism |
| TB | InhA, KatG | 1.6M | Novel antibiotic design |

*Part of [Quantum for Humanity](https://github.com/vivekiniitm-stack/Quantum-for-humanity)*